In [9]:
import tensorflow as tf
import numpy as np

# --- MODIFICAÇÃO de otimização: Ativação da Política de Mixed Precision ---
# Isso fará com que o processamento aproveite os Tensor Cores 
from tensorflow.keras import mixed_precision
policy = mixed_precision.Policy('float32')
mixed_precision.set_global_policy(policy)

# %load_ext autoreload
# %autoreload 2

import os
import sys
import time
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras import Model
import load_model_compat

In [10]:
### At this threshold the FPR of the model we trained is 0.1
thr = 0.08106673508882523 

In [11]:
print(tf.config.list_physical_devices('GPU'))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [12]:
attack_types = ['FTP-Patator','SSH-Patator','Slowloris','Slowhttptest','Hulk','GoldenEye','Heartbleed', 'Web-Attack', 'Infiltration','Botnet','PortScan','DDoS']

In [13]:
attack_to_label={
    'FTP-Patator':1,'SSH-Patator':2, #Tuesday attacks
    'Slowloris':3,'Slowhttptest':4,'Hulk':5,'GoldenEye':6,'Heartbleed':7, #Wednesday attacks
    'Web-Attack':8, 'Infiltration':9, #Thursday attacks
    'Botnet':10,'PortScan':11,'DDoS':12 #Friday attacks
}

In [14]:
def get_files(day,prefix = '../data/packet_based/'):
    all_files = []
    prefix = prefix+day
    for file in os.listdir(prefix):
        if file.endswith(".npy") and file.startswith('part'):
            all_files.append(os.path.join(prefix, file))
    all_files = sorted(all_files)
    return all_files

In [15]:
def get_test_set(day):
    train_min = np.load('../data/packet_based/x_train_meta/train_min.npy')
    train_max = np.load('../data/packet_based/x_train_meta/train_max.npy')
    
    all_files = get_files(day)
    x_test = []
    for f in all_files:
        print (f)
        x_test.append(np.load(f))
    x_test = np.concatenate(x_test,axis=0)

    yt = np.load('../data/packet_based/'+day+'/labels.npy')
    y_test = yt
    return x_test,y_test,train_min,train_max

In [16]:
@tf.function
def test_step(x):
    def_mask = tf.random.uniform(shape=[1*100,timesteps,num_input])
    def_mask = tf.cast((def_mask>0.75),tf.float32)
    x_normalized =(x - train_min)/(train_max - train_min+0.000001)

    partial_x = def_mask*x_normalized
    rec_x = model(partial_x, training=False)
    score = tf.reduce_mean(tf.square(rec_x - x_normalized),axis=[1,2])
    score = tf.reduce_min(tf.reshape(score,[5,20]),axis=-1)
    score = tf.reduce_sum(score)
    return score

In [17]:
attack_type = 'Slowloris'

In [18]:
if attack_type in ['FTP-Patator','SSH-Patator']:
    day = 'tuesday'
elif attack_type in ['Slowloris','Slowhttptest','Hulk','GoldenEye','Heartbleed']:
    day = 'wednesday'
elif attack_type in ['Web-Attack', 'Infiltration']:
    day = 'thursday'
else:
    day = 'friday'

In [19]:
x_test,y_test,train_min,train_max = get_test_set(day)
timesteps = 20
num_input = x_test.shape[1]
print(x_test.shape,y_test.shape)

model = load_model_compat.load_model('../models/pkt_model/')

../data/packet_based/wednesday/part_00000.npy
../data/packet_based/wednesday/part_00001.npy
../data/packet_based/wednesday/part_00002.npy
../data/packet_based/wednesday/part_00003.npy
../data/packet_based/wednesday/part_00004.npy
../data/packet_based/wednesday/part_00005.npy
../data/packet_based/wednesday/part_00006.npy
../data/packet_based/wednesday/part_00007.npy
../data/packet_based/wednesday/part_00008.npy
../data/packet_based/wednesday/part_00009.npy
../data/packet_based/wednesday/part_00010.npy
../data/packet_based/wednesday/part_00011.npy
../data/packet_based/wednesday/part_00012.npy
../data/packet_based/wednesday/part_00013.npy
../data/packet_based/wednesday/part_00014.npy
../data/packet_based/wednesday/part_00015.npy
../data/packet_based/wednesday/part_00016.npy
../data/packet_based/wednesday/part_00017.npy
../data/packet_based/wednesday/part_00018.npy
../data/packet_based/wednesday/part_00019.npy
../data/packet_based/wednesday/part_00020.npy
../data/packet_based/wednesday/par

In [20]:
attack_label = attack_to_label[attack_type]
x_test_mal = x_test[y_test==attack_label]
x_test_mal = np.concatenate((np.zeros((timesteps-1,num_input)),x_test_mal),axis=0)
print (x_test_mal.shape)
x_test_mal = x_test_mal[:25000].astype(np.float32)
score_np = np.zeros(len(x_test_mal))
st = timesteps-1
begin_time = time.time()
for i in range(len(x_test_mal)-timesteps):
    if i%10000==0:
        print (i,time.time() - begin_time)
    sample = x_test_mal[i:i+timesteps][None]
    score_temp = test_step(sample)
    score_np[st+i] = score_temp.numpy()
print (i,time.time() - begin_time)

mal_scores = score_np[timesteps:]

normal_tpr = np.sum(mal_scores>=thr)/len(mal_scores)
print ("TPR in normal setting for "+attack_type+" is {0:0.4f}".format(normal_tpr))

(47605, 29)
0 0.0001556873321533203
10000 15.30620002746582
20000 29.895095586776733
24979 37.119590282440186
TPR in normal setting for Slowloris is 0.8849


# Crafting Adversarial Examples

In [21]:
alpha_delay = tf.Variable(np.zeros((1, 1,num_input),dtype=np.float32),name='delay')
alpha_split = tf.Variable(np.zeros((1),dtype=np.float32),name='split')
alpha_inject = tf.Variable(np.zeros((1, 1,num_input), dtype=np.float32),name='modifier')

In [22]:
optimizer_split = tf.keras.optimizers.Adam(learning_rate=0.1)
optimizer_inject = tf.keras.optimizers.Adam(learning_rate=0.1)

In [23]:
@tf.function
def get_delayed_splited(x,p_len):
    alpha_split_2 = tf.zeros((1,1,num_input)) + alpha_split
    alpha_split_2 = tf.minimum(alpha_split_2,0.)
    alpha_split_2 = tf.maximum(alpha_split_2,-p_len+np.float32(61))
    alpha_delay_2 = tf.maximum(alpha_delay,0.)
    alpha_delay_2 = tf.minimum(alpha_delay_2,15.)
    mask = np.ones((1,1,29))
    masked_alpha = alpha_delay_2*mask
    mask_split = np.zeros((1,1,29))
    mask_split[0,0,1] = mask_split[0,0,3] = 1
    mask_split = mask_split.astype(np.bool)
    alpha_final = tf.where(mask_split,alpha_split_2,masked_alpha)

    last_ts_modified = x[0,-1]+alpha_final
    adv_x = tf.concat((x[:,:19],last_ts_modified),axis=1)
    return adv_x

In [24]:
@tf.function
def delay_split_optim(x,p_len):
    with tf.GradientTape() as tape:
        adv_x = get_delayed_splited(x,p_len)
        adv_x_normalized = (adv_x- train_min)/(train_max - train_min+0.000001)
        rand_mask = tf.random.uniform(shape=[100,timesteps,num_input])
        rand_mask = tf.cast((rand_mask>0.75),tf.float32)
        partial_adv_x_n = adv_x_normalized*rand_mask
        rec_adv_x_n = model(partial_adv_x_n,training=False)
        score1_split = tf.reduce_mean(tf.square(rec_adv_x_n - adv_x_normalized),axis=[1,2])
        score1_split = tf.reduce_sum(score1_split)
        loss_split = score1_split
        
    gradients = tape.gradient(loss_split, [alpha_delay,alpha_split])
    optimizer_split.apply_gradients(zip(gradients, [alpha_delay,alpha_split]))


In [25]:
@tf.function
def get_injected(x,inject_mask):
    packet_mins = [0,60,20,0,0,0,0,1,1,1,1,1,0,0,0,0,0,0,0,0,0,0,1,1,0,1,1,0,1]
    packet_maxs = [15,2**16,20,2**16,1,1,1,255,2**16,2**16,2**32,2**32,1,1,1,1,1,1,1,1,1,1,2**16,2**16,2**16,2**16,2**16,0,1]
    alpha_inject_2 = tf.minimum(alpha_inject,packet_maxs)
    alpha_inject_2 = tf.maximum(alpha_inject_2,packet_mins)
    alpha_inject_masked= alpha_inject_2*inject_mask
    adv_x1 = tf.concat((x[:,:19],alpha_inject_masked),axis=1)
    adv_x2 = tf.concat((x[:,1:19],alpha_inject_masked,x[:,19:]),axis=1)
    return adv_x1,adv_x2

In [26]:
@tf.function
def inject_optim(x,inject_mask):
    with tf.GradientTape() as tape:
        adv_x1,adv_x2 = get_injected(x,inject_mask)
        adv_x1_normalized = (adv_x1- train_min)/(train_max - train_min+0.000001)
        adv_x2_normalized = (adv_x2- train_min)/(train_max - train_min+0.000001)
        rand_mask = tf.random.uniform(shape=[100,timesteps,num_input])
        rand_mask = tf.cast((rand_mask>0.75),tf.float32)
        partial_adv_x_n1 = adv_x1_normalized*rand_mask
        rand_mask = tf.random.uniform(shape=[100,timesteps,num_input])
        rand_mask = tf.cast((rand_mask>0.75),tf.float32)
        partial_adv_x_n2 = adv_x2_normalized*rand_mask
        rec_adv_x_n1 = model(partial_adv_x_n1,training=False)
        rec_adv_x_n2 = model(partial_adv_x_n2,training=False)
        score1_inject1 = tf.reduce_mean(tf.square(rec_adv_x_n1 - adv_x1_normalized),axis=[1,2])
        score1_inject1 = tf.reduce_sum(score1_inject1)
        score1_inject2 = tf.reduce_mean(tf.square(rec_adv_x_n2 - adv_x2_normalized),axis=[1,2])
        score1_inject2 = tf.reduce_sum(score1_inject2)
        loss_inject = score1_inject1 + score1_inject2

    gradients = tape.gradient(loss_inject, [alpha_inject])
    optimizer_inject.apply_gradients(zip(gradients, [alpha_inject]))

In [27]:
def find_adv(x):
    sc = test_step(x)
    sc = sc.numpy()
    if sc < thr:
        return 'cons_as_ben'
    if x[0,-1,-1] == 2: #packet is sent from victim
        return inject(np.copy(x))
    
    #packet is sent from attacker
    res = delay_and_split(np.copy(x))
    if res and len(res) > 0:
        return ('split', res)
    i_res = inject(np.copy(x))
    return i_res


In [28]:
# def delay_and_split(x):
#     alpha_delay.assign(np.zeros(alpha_delay.shape))
#     alpha_split.assign(np.zeros(alpha_split.shape))
#     len_last = x[0,-1,1]
#     ip_len_last = x[0,-1,3]
#     adv_x = get_delayed_splited(x,len_last)
#     adv_x = adv_x.numpy()
#     sc = test_step(adv_x)
#     sc = sc.numpy()
#     if sc<thr:
#         return [adv_x]
#     res = []
#     for i in range(300):
#         delay_split_optim(x,len_last)
#         adv_x = get_delayed_splited(x,len_last)
#         adv_x = adv_x.numpy()
#         adv_x[0,-1,1:4] = np.round(adv_x[0,-1,1:4])
#         sc = test_step(adv_x)
#         sc = sc.numpy()
#         if sc<thr:
#             first_part = np.copy(adv_x)
#             diff = len_last - adv_x[0,-1,1] 
#             if diff>0:
#                 adv_x[0,-1,1] = diff + 60
#                 adv_x[0,-1,0] = 0
#                 adv_x[0,-1,3] = diff + 60 - 14 #14 is the frame header len.
#                 second_part = delay_and_split(adv_x)
#                 if second_part==None:
#                     return None
#                 res.append(first_part)
#                 res.extend(second_part)
#             else:
#                 res.append(first_part)
#             break
#     if len(res)==0:
#         return None
#     return res

# --- MODIFICAÇÃO de otimização: Compilação para GPU do delay_and_split ---
@tf.function
def compiled_delay_loop(x, len_last, thr_val):
    """Roda as 300 iterações de gradiente para delay_split 100% na GPU"""
    alpha_delay.assign(tf.zeros_like(alpha_delay))
    alpha_split.assign(tf.zeros_like(alpha_split))
    
    def cond(i, fooled, adv_x_res):
        return tf.logical_and(i < 300, tf.logical_not(fooled))
    
    def body(i, fooled, adv_x_res):
        delay_split_optim(x, len_last)
        ax = get_delayed_splited(x, len_last)
        
        # Arredondamento nativo na GPU para índices 1, 2 e 3
        ax_round = tf.concat([
            ax[:, :, :1], 
            tf.math.round(ax[:, :, 1:4]), 
            ax[:, :, 4:]
        ], axis=-1)
        
        sc = test_step(ax_round)
        is_fooled = (sc < thr_val)
        return i + 1, is_fooled, ax_round

    init_ax = get_delayed_splited(x, len_last)
    _, fooled, final_adv_x = tf.while_loop(
        cond, body,
        loop_vars=[tf.constant(0), tf.constant(False), init_ax],
        parallel_iterations=1
    )
    return fooled, final_adv_x

def delay_and_split(x):
    len_last = x[0, -1, 1]
    
    # Checagem preliminar para evitar loops desnecessários
    init_ax = get_delayed_splited(x, len_last).numpy()
    if test_step(init_ax).numpy() < thr:
        return [init_ax]
        
    # Aciona o loop na GPU em alta velocidade
    fooled, final_adv_x_tf = compiled_delay_loop(x, len_last, tf.constant(thr, dtype=tf.float32))
    
    res = []
    
    if fooled.numpy():
        adv_x = final_adv_x_tf.numpy()
        first_part = np.copy(adv_x)
        diff = len_last - adv_x[0, -1, 1]
        
        if diff > 0:
            adv_x[0, -1, 1] = diff + 60
            adv_x[0, -1, 0] = 0
            adv_x[0, -1, 3] = diff + 60 - 14 # 14 é o tamanho do cabeçalho do frame
            
            second_part = delay_and_split(adv_x)
            if second_part is None:
                return None
            res.append(first_part)
            res.extend(second_part)
        else:
            res.append(first_part)
            
    if len(res) == 0:
        return None
    return res

In [29]:
tcp_mask = [1]*8 + [1]*16 + [0]*3 + [0]*1 + [1]*1
udp_mask = [1]*8 + [0]*16 + [1]*3 + [0]*1 + [1]*16
# def inject(x,mask_type = 'tcp'):
#     alpha_inject.assign(np.zeros(alpha_inject.shape))
#     cur_mask = tcp_mask if mask_type=='tcp' else udp_mask
#     for i in range(300):
#         inject_optim(x,cur_mask)
#         adv_x1,adv_x2 = get_injected(x,cur_mask)
#         adv_x1,adv_x2  = adv_x1.numpy(),adv_x2.numpy()
#         adv_x1[0,:,1:] = np.round(adv_x1[0,:,1:])
#         sc = test_step(adv_x1)
#         sc = sc.numpy()
#         adv_x2[0,:,1:] = np.round(adv_x2[0,:,1:])
#         sc2 = test_step(adv_x2)
#         sc2 = sc2.numpy()
#         if sc<thr and sc2<thr: #fooled
#             fake_packets.append(adv_x1[0,-1])
#             return ('inject',adv_x1[0,-1]) #<--- the packet which is inject should be returned
#     res = None
#     return res

# --- MODIFICAÇÃO de otimização: Compilação para GPU do inject_loop ---
@tf.function
def compiled_inject_loop(x, cur_mask, thr_val):
    alpha_inject.assign(tf.zeros_like(alpha_inject))
    
    def cond(i, fooled, adv_x1_res, adv_x2_res):
        return tf.logical_and(i < 300, tf.logical_not(fooled))
    
    def body(i, fooled, adv_x1_res, adv_x2_res):
        inject_optim(x, cur_mask)
        ax1, ax2 = get_injected(x, cur_mask)
        
        # Arredondamento nativo no TF
        ax1_round = tf.concat([ax1[:, :, :1], tf.math.round(ax1[:, :, 1:])], axis=-1)
        ax2_round = tf.concat([ax2[:, :, :1], tf.math.round(ax2[:, :, 1:])], axis=-1)
        
        sc = test_step(ax1_round)
        sc2 = test_step(ax2_round)
        
        is_fooled = tf.logical_and(sc < thr_val, sc2 < thr_val)
        return i + 1, is_fooled, ax1_round, ax2_round

    init_ax1, init_ax2 = get_injected(x, cur_mask)
    
    _, fooled, final_ax1, _ = tf.while_loop(
        cond, body, 
        loop_vars=[tf.constant(0), tf.constant(False), init_ax1, init_ax2],
        parallel_iterations=1
    )
    return fooled, final_ax1[0, -1]

def inject(x, mask_type='tcp'):
    cur_mask = tcp_mask if mask_type == 'tcp' else udp_mask
    
    # Passamos o threshold como constante do TF
    fooled, fake_packet = compiled_inject_loop(x, cur_mask, tf.constant(thr, dtype=tf.float32))
    
    if fooled.numpy():
        p_numpy = fake_packet.numpy()
        fake_packets.append(p_numpy)
        return ('inject', p_numpy)
    return None

Iniciando o teste rápido (somente Inject) para 100 pacotes...
Pkt: 10/100 | Vel: 85.99 pkt/s | Tempo decorrido: 0.12 s | Mal: 0 | Ben: 10 | Fooled: 0
Pkt: 20/100 | Vel: 2.35 pkt/s | Tempo decorrido: 8.51 s | Mal: 3 | Ben: 16 | Fooled: 1
Pkt: 30/100 | Vel: 3.52 pkt/s | Tempo decorrido: 8.53 s | Mal: 3 | Ben: 26 | Fooled: 1
Pkt: 40/100 | Vel: 4.68 pkt/s | Tempo decorrido: 8.55 s | Mal: 3 | Ben: 36 | Fooled: 1
Pkt: 50/100 | Vel: 2.34 pkt/s | Tempo decorrido: 21.38 s | Mal: 8 | Ben: 41 | Fooled: 1
Pkt: 60/100 | Vel: 1.33 pkt/s | Tempo decorrido: 44.96 s | Mal: 18 | Ben: 41 | Fooled: 1
Pkt: 70/100 | Vel: 1.03 pkt/s | Tempo decorrido: 67.91 s | Mal: 28 | Ben: 41 | Fooled: 1
Pkt: 80/100 | Vel: 0.88 pkt/s | Tempo decorrido: 90.71 s | Mal: 38 | Ben: 41 | Fooled: 1
Pkt: 90/100 | Vel: 0.86 pkt/s | Tempo decorrido: 104.36 s | Mal: 43 | Ben: 44 | Fooled: 3
Pkt: 100/100 | Vel: 0.79 pkt/s | Tempo decorrido: 127.25 s | Mal: 53 | Ben: 44 | Fooled: 3

Teste de 100 pacotes concluído em 127.25 segundos.


Iniciando o teste rápido (somente Inject) para 100 pacotes...
Pkt: 10/100 | Vel: 43.30 pkt/s | Tempo decorrido: 0.23 s | Mal: 0 | Ben: 10 | Fooled: 0
Pkt: 20/100 | Vel: 1.15 pkt/s | Tempo decorrido: 17.39 s | Mal: 4 | Ben: 15 | Fooled: 1
Pkt: 30/100 | Vel: 1.72 pkt/s | Tempo decorrido: 17.40 s | Mal: 4 | Ben: 25 | Fooled: 1
Pkt: 40/100 | Vel: 2.30 pkt/s | Tempo decorrido: 17.43 s | Mal: 4 | Ben: 35 | Fooled: 1
Pkt: 50/100 | Vel: 1.31 pkt/s | Tempo decorrido: 38.08 s | Mal: 9 | Ben: 40 | Fooled: 1
Pkt: 60/100 | Vel: 0.71 pkt/s | Tempo decorrido: 84.59 s | Mal: 19 | Ben: 40 | Fooled: 1
Pkt: 70/100 | Vel: 0.51 pkt/s | Tempo decorrido: 136.62 s | Mal: 29 | Ben: 40 | Fooled: 1
Pkt: 80/100 | Vel: 0.41 pkt/s | Tempo decorrido: 193.40 s | Mal: 39 | Ben: 40 | Fooled: 1
Pkt: 90/100 | Vel: 0.38 pkt/s | Tempo decorrido: 237.95 s | Mal: 46 | Ben: 40 | Fooled: 4
Pkt: 100/100 | Vel: 0.35 pkt/s | Tempo decorrido: 282.30 s | Mal: 56 | Ben: 40 | Fooled: 4

Teste de 100 pacotes concluído em 282.30 segundos.

In [30]:
stream = []
stream_status = []
fake_packets = []
cons_as_mal = 0
cons_as_ben = 0
fooled = 0
for i in range(timesteps-1):
    stream.append(x_test_mal[i])
    stream_status.append(None)
    
total_pacotes = len(x_test_mal)
pacotes_processados = 0    
begin_time = time.time()
log_steps = 100  # Log every 'log_steps' packets

print(f"Starting adversarial testing for {attack_type}...")

for i in range(timesteps-1, len(x_test_mal)):
    
    x = np.zeros((1,20,29), dtype=np.float32)
    x[0,:19] = np.array(stream[-19:])
    x[0,19] = x_test_mal[i]
    
    temp = find_adv(np.copy(x))
    stream_status.append(temp)
    
    if isinstance(temp, type(None)):
        stream.append(x_test_mal[i])
        cons_as_mal += 1
    elif temp == 'cons_as_ben':
        stream.append(x_test_mal[i])
        cons_as_ben += 1
    elif temp[0] == 'split':
        fooled += 1
        for pkt in temp[1]:
            p2 = pkt[0,-1]
            stream.append(p2)
    elif temp[0] == 'inject':
        fooled += 1
        fake_pkt = temp[1]
        stream.append(fake_pkt)
        stream.append(x_test_mal[i])
        
    pacotes_processados += 1
    
    if pacotes_processados % log_steps == 0:
        elapsed = time.time() - begin_time
        vel = pacotes_processados / elapsed
        eta_hours = (total_pacotes - pacotes_processados) / vel / 3600
        
        print(f"Pkt: {pacotes_processados}/{total_pacotes} | "
              f"Vel: {vel:.2f} pkt/s | "
              f"Tempo decorrido: {elapsed:.2f} s | "
              f"Mal: {cons_as_mal} | Ben: {cons_as_ben} | Fooled: {fooled} | " 
              f"ETA: {eta_hours:.2f} h")

    # # Condição de parada do teste
    # if pacotes_processados >= total_pacotes:
    #     break



print("-" * 80)
print(f"Processamento de {pacotes_processados} pacotes concluído em {(time.time() - begin_time)/60:.2f} minutos.")

Starting adversarial testing for Slowloris...
Pkt: 100/25000 | Vel: 1.03 pkt/s | Tempo decorrido: 96.90 s | Mal: 88 | Ben: 11 | Fooled: 1 | ETA: 6.70 h
Pkt: 200/25000 | Vel: 1.00 pkt/s | Tempo decorrido: 199.50 s | Mal: 188 | Ben: 11 | Fooled: 1 | ETA: 6.87 h
Pkt: 300/25000 | Vel: 0.99 pkt/s | Tempo decorrido: 301.51 s | Mal: 288 | Ben: 11 | Fooled: 1 | ETA: 6.90 h
Pkt: 400/25000 | Vel: 0.99 pkt/s | Tempo decorrido: 402.71 s | Mal: 388 | Ben: 11 | Fooled: 1 | ETA: 6.88 h
Pkt: 500/25000 | Vel: 0.99 pkt/s | Tempo decorrido: 505.25 s | Mal: 488 | Ben: 11 | Fooled: 1 | ETA: 6.88 h
Pkt: 600/25000 | Vel: 0.99 pkt/s | Tempo decorrido: 608.53 s | Mal: 588 | Ben: 11 | Fooled: 1 | ETA: 6.87 h
Pkt: 700/25000 | Vel: 0.98 pkt/s | Tempo decorrido: 714.81 s | Mal: 688 | Ben: 11 | Fooled: 1 | ETA: 6.89 h
Pkt: 800/25000 | Vel: 0.95 pkt/s | Tempo decorrido: 840.86 s | Mal: 788 | Ben: 11 | Fooled: 1 | ETA: 7.07 h
Pkt: 900/25000 | Vel: 0.95 pkt/s | Tempo decorrido: 944.97 s | Mal: 888 | Ben: 11 | Fooled: 

In [31]:
print ("TPR in adversarial setting for "+attack_type+" is {0:0.4f}".format(cons_as_mal/len(x_test_mal)))

TPR in adversarial setting for Slowloris is 0.8736


In [32]:
from experiment_results import ExperimentResult, save_result_json
results_path = "../results/packet_based_adversarial_results.json"
save_result_json(
    ExperimentResult(
        best_threshold=float(thr),
        victim_tpr=float(cons_as_mal / len(x_test_mal)),
        attacker_tpr=float(np.sum(mal_scores >= thr) / len(mal_scores)),
        total_runtime_seconds=float(time.time() - begin_time),
        attack_type=str(attack_type),
        model_type="packet",
        normal_tpr=normal_tpr,
    ),
    results_path,
)
print("Adversarial result saved to ../results/packet_based_adversarial_results.json")

Adversarial result saved to ../results/packet_based_adversarial_results.json
